# Burrows–Wheeler Transform for DNA Sequence Searching

## 1. Introduction

The aim of this practical is to implement the Burrows–Wheeler Transform (BWT) and its inverse in Python, and to investigate the occurrence of a specific DNA sequence and its reverse complement in a reference genome sequence.

## Background

Searching for a specific DNA sequence is a common task in bioinformatics. A straightforward approach is to compare the target sequence directly with the reference sequence. However, repeated searches in large genomic datasets can be computationally expensive.

The Burrows–Wheeler Transform rearranges a sequence into a representation that supports efficient indexing and pattern searching. It is an important component of the FM-index, which is used in bioinformatics sequence-searching and alignment algorithms.

The BWT is reversible when the transformed sequence includes a unique end-of-sequence marker. This allows the original sequence to be reconstructed from its transformed representation.

## Objectives

1. Implement the Burrows–Wheeler Transform in Python.
2. Implement the inverse transform and verify that it reconstructs the original sequence.
3. Count occurrences of `AGATA` in the reference using backward search on the BWT.
4. Count occurrences of its reverse complement in the same way.
5. Check the BWT counts against a direct search and discuss the limitations of the implementation.

## Approach

The transformation will first be explored using a short example so that the rotation, sorting and last-column extraction steps can be understood. The implementation will then be tested before being applied to the reference sequence. The BWT is then used to count the pattern with backward search, and the counts are checked with a direct search.


## 2. Loading and Validating the Reference Sequence

The reference sequence is provided in FASTA format. A FASTA file contains a header line, beginning with the character `>`, followed by the nucleotide sequence. The sequence may be split across multiple lines.

Before applying the Burrows–Wheeler Transform, the reference must be extracted and validated. The header and whitespace should not be included in the DNA sequence.

Validation is important because the transformation and subsequent pattern searches must operate on the biological sequence itself, rather than on file-format information.


In [2]:
# Read the reference sequence from the FASTA file.

# This file is provided by the course and is not included in this repository.
fasta_path = "/home/jovyan/shared-team/SL50188A/bwt/ref-for-bwt-search.fa"

with open(fasta_path, "r") as fasta_file:
    lines = fasta_file.readlines()

# Extract the header and sequence separately.
header = None
sequence_lines = []

for line in lines:
    line = line.strip()

    # Ignore blank lines.
    if not line:
        continue

    # A FASTA header begins with '>'.
    if line.startswith(">"):
        if header is not None:
            raise ValueError("Expected a single reference sequence.")
        header = line[1:]
    else:
        # Collect sequence lines, which may span multiple lines.
        sequence_lines.append(line)

# Combine the sequence lines into one continuous string.
reference = "".join(sequence_lines).upper()

# Validate the input before analysing it.
if header is None:
    raise ValueError("No FASTA header was found.")

if not reference:
    raise ValueError("The reference sequence is empty.")

if any(base not in "ACGT" for base in reference):
    raise ValueError("The reference contains unexpected characters.")

# Display a summary of the input.
print("Reference ID:", header)
print("Sequence length:", len(reference), "bases")
print("Unique nucleotide bases:", sorted(set(reference)))
print("First 60 bases:", reference[:60])


Reference ID: zamsref
Sequence length: 1989 bases
Unique nucleotide bases: ['A', 'C', 'G', 'T']
First 60 bases: GGCCAGCCGCTATTGCGATCATCTGGTGGTACTGGCAAGCGGTCGCGTGATGGCGCAAGG


## 3. Understanding the Burrows–Wheeler Transform

The Burrows–Wheeler Transform (BWT) rearranges a string into a representation that is useful for sequence indexing and searching. It is reversible when a unique end-of-sequence marker is included.

For this implementation, the special character `$` marks the end of the sequence. It must occur exactly once and is treated as lexicographically smaller than the DNA bases.

The transformation involves three main steps:

1. Add `$` to the end of the sequence, then generate all cyclic rotations of the result.
2. Sort the rotations in lexicographical order.
3. Extract the final character from each sorted rotation to construct the BWT.

A small example is used first to make each step explicit. The same algorithm can then be applied to a longer DNA reference sequence.


In [54]:
# Worked example: rotations of GAT with the end marker added
text = "GAT$"

# Make every rotation
rotations = []
for i in range(len(text)):
    rotations.append(text[i:] + text[:i])

# Sort them
sorted_rotations = sorted(rotations)
for row in sorted_rotations:
    print(row)

# F is the first letter of each row, L is the last letter
F = ""
L = ""
for row in sorted_rotations:
    F = F + row[0]
    L = L + row[-1]
print("F:", F)
print("L (the BWT):", L)

$GAT
AT$G
GAT$
T$GA
F: $AGT
L (the BWT): TG$A


## 4. Implementing the Burrows–Wheeler Transform

The previous example demonstrated how the BWT is constructed by generating all cyclic rotations of a sequence, sorting these rotations lexicographically, and extracting the final character from each sorted row.

In this section, these steps are implemented in a reusable Python function. The function appends a unique end-of-sequence marker (`$`) before constructing the rotations. This marker identifies the end of the original sequence and makes the transformation reversible.

The implementation accepts DNA sequences containing A, C, G and T. It also validates the input to help prevent errors. The function will first be tested on the short sequence `GAT`, for which the expected BWT is `TG$A`.


In [26]:
def burrows_wheeler_transform(sequence):
    """Return the Burrows–Wheeler Transform of a DNA sequence."""

    sequence = sequence.upper()

    # Validate the input sequence
    if not sequence:
        raise ValueError("The DNA sequence must not be empty.")

    if "$" in sequence:
        raise ValueError("The input must not already contain the '$' marker.")

    if any(base not in "ACGT" for base in sequence):
        raise ValueError("The sequence must contain only A, C, G and T.")

    # Add a unique end-of-sequence marker
    text = sequence + "$"

    # Generate all cyclic rotations
    rotations = [
        text[i:] + text[:i]
        for i in range(len(text))
    ]

    # Sort rotations in lexicographical order
    rotations.sort()

    # Extract the last character from each sorted rotation
    bwt = "".join(row[-1] for row in rotations)

    return bwt


# Test the function using the example from Section 3
example_bwt = burrows_wheeler_transform("GAT")

print("Original sequence:", "GAT")
print("BWT:", example_bwt)

assert example_bwt == "TG$A", "The BWT test failed."
print("Test passed.")

Original sequence: GAT
BWT: TG$A
Test passed.


## 5. Implementing the Inverse Burrows–Wheeler Transform

The Burrows–Wheeler Transform rearranges a sequence into a form that can support efficient sequence indexing. To recover the original sequence, we need to implement the inverse Burrows–Wheeler Transform.

The inverse transform uses the relationship between the last column (L) and the first column (F) of the sorted rotation matrix. The first column can be obtained by sorting the characters of the BWT string. By tracking the correspondence between these columns, the original sequence can be reconstructed.

In this section, the inverse transform will be implemented and tested using the short DNA sequence from the previous section. Successful reconstruction will provide evidence that the forward and inverse transformations are consistent.

In [60]:
def inverse_burrows_wheeler_transform(bwt):
    """Reconstruct the original DNA sequence from its BWT."""

    # Check that the BWT contains exactly one end-of-sequence marker.
    if bwt.count("$") != 1:
        raise ValueError("The BWT must contain exactly one '$' marker.")

    # Start with an empty table.
    table = [""] * len(bwt)

    # Each pass puts the BWT letters in front of the current rows and sorts.
    # After len(bwt) passes, every row is a full rotation of the original text.
    for _ in range(len(bwt)):
        table = sorted(
            bwt[i] + table[i]
            for i in range(len(bwt))
        )

    # Find the row ending with the end-of-sequence marker.
    for row in table:
        if row.endswith("$"):
            return row[:-1]

    raise ValueError("Could not reconstruct the original sequence.")


# Test the inverse transform using the BWT from Section 4.
reconstructed_sequence = inverse_burrows_wheeler_transform(example_bwt)

print("BWT:", example_bwt)
print("Reconstructed sequence:", reconstructed_sequence)

# Check that the original sequence is recovered.
assert reconstructed_sequence == "GAT", "The inverse BWT test failed."

print("Inverse BWT test passed.")

BWT: TG$A
Reconstructed sequence: GAT
Inverse BWT test passed.



## 6. Applying and Validating the BWT on the Reference Sequence

After testing the forward and inverse BWT implementations on a short example, we now apply them to the reference DNA sequence loaded from the FASTA file.

The transformed sequence is reconstructed using the inverse BWT. Comparing the reconstructed sequence with the original reference verifies that the two implementations work consistently on the input data.

This validation is important before proceeding to DNA pattern searching, because errors in the transformation or reconstruction could affect subsequent results.


In [30]:
# Calculate the BWT of the reference DNA sequence.
reference_bwt = burrows_wheeler_transform(reference)

print("Reference sequence length:", len(reference))
print("BWT length:", len(reference_bwt))

# Reconstruct the original reference sequence.
reconstructed_reference = inverse_burrows_wheeler_transform(reference_bwt)

# Check whether reconstruction matches the original sequence.
assert reconstructed_reference == reference, (
    "The reconstructed sequence does not match the reference."
)

print("Reconstruction successful:", reconstructed_reference == reference)
print("Validation passed.")

Reference sequence length: 1989
BWT length: 1990
Reconstruction successful: True
Validation passed.



## 7. Direct Search for the Target Pattern

The next step is to identify how many times the target DNA pattern `AGATA` occurs in the reference sequence.

A straightforward approach is to examine every possible starting position and compare the subsequence at that position with the target pattern. This method identifies overlapping occurrences as well as non-overlapping ones.

Although direct searching is simple and suitable for a short sequence, more advanced methods, such as BWT-based backward searching, can support efficient pattern searching in large indexed genomic datasets.

In this section, direct searching is used as a baseline for validating the results of subsequent searches.

This direct search is a check. The count that answers the question comes from the BWT search in Section 10


In [31]:
# Define the target DNA pattern.
pattern = "AGATA"

# Store the starting positions of all occurrences.
pattern_positions = []

# Check every possible starting position, including overlapping matches.
for i in range(len(reference) - len(pattern) + 1):
    if reference[i:i + len(pattern)] == pattern:
        pattern_positions.append(i)

# Display the search results.
print("Target pattern:", pattern)
print("Number of occurrences:", len(pattern_positions))
print("Zero-based starting positions:", pattern_positions)

# Confirm that the reported positions contain the target pattern.
assert all(
    reference[i:i + len(pattern)] == pattern
    for i in pattern_positions
)

print("Pattern search validation passed.")


Target pattern: AGATA
Number of occurrences: 5
Zero-based starting positions: [200, 205, 809, 917, 921]
Pattern search validation passed.



## 8. Direct Search for the Reverse Complement

DNA consists of two complementary strands that run in opposite directions. In DNA, adenine (A) pairs with thymine (T), while cytosine (C) pairs with guanine (G).

To search for a DNA pattern on the opposite strand, we calculate its reverse complement. This involves replacing each nucleotide with its complementary base and then reversing the resulting sequence to represent it in the conventional 5′ to 3′ direction.

For the target pattern `AGATA`, the complementary sequence is `TCTAT`, and its reverse complement is `TATCT`.

In this section, we calculate the reverse complement programmatically and search for its occurrences in the reference sequence using the same direct-search method as in Section 7.


In [32]:
# Define the complementary DNA bases.
complement_table = str.maketrans("ACGT", "TGCA")

# Calculate the reverse complement of the target pattern.
reverse_complement = pattern.translate(complement_table)[::-1]

print("Original pattern:", pattern)
print("Reverse complement:", reverse_complement)

# Store the starting positions of all reverse-complement occurrences.
reverse_complement_positions = []

# Search the reference sequence at every possible starting position.
for i in range(len(reference) - len(reverse_complement) + 1):
    if reference[i:i + len(reverse_complement)] == reverse_complement:
        reverse_complement_positions.append(i)

# Display the search results.
print("Number of reverse-complement occurrences:",
      len(reverse_complement_positions))
print("Zero-based starting positions:",
      reverse_complement_positions)

# Validate the reported positions.
assert all(
    reference[i:i + len(reverse_complement)] == reverse_complement
    for i in reverse_complement_positions
)

print("Reverse-complement search validation passed.")


Original pattern: AGATA
Reverse complement: TATCT
Number of reverse-complement occurrences: 3
Zero-based starting positions: [137, 1125, 1257]
Reverse-complement search validation passed.



## 9. Summary of the Direct Searches

The target pattern and its reverse complement were searched independently in the reference DNA sequence using direct string matching.

A summary table is used to compare the patterns, their occurrence counts, and their zero-based starting positions. Searching both orientations is important because a DNA sequence of interest may be represented on either strand.

The results from direct searching will also provide a baseline for validating a more advanced BWT-based search implementation.


In [35]:
# Summarise the target pattern and reverse-complement search results.
search_summary = [
    {
        "Pattern type": "Target pattern",
        "Sequence": pattern,
        "Occurrences": len(pattern_positions),
        "Zero-based positions": pattern_positions
    },
    {
        "Pattern type": "Reverse complement",
        "Sequence": reverse_complement,
        "Occurrences": len(reverse_complement_positions),
        "Zero-based positions": reverse_complement_positions
    }
]

# Display the summary.
for result in search_summary:
    print("\nPattern type:", result["Pattern type"])
    print("Sequence:", result["Sequence"])
    print("Number of occurrences:", result["Occurrences"])
    print("Zero-based positions:", result["Zero-based positions"])


Pattern type: Target pattern
Sequence: AGATA
Number of occurrences: 5
Zero-based positions: [200, 205, 809, 917, 921]

Pattern type: Reverse complement
Sequence: TATCT
Number of occurrences: 3
Zero-based positions: [137, 1125, 1257]


## 10. Searching with the BWT (backward search)

To search with the BWT, I need two helper pieces: where each letter starts in F, and a way to count letters in L.

1. F is the BWT's letters sorted.
2. Rows beginning with the same letter sit together in F.
3. Backward search starts from one of these groups of rows, so I need to know where each group begins.

The function below finds the first row of each letter in F. I tested it on the BWT of GAT (`TG$A`), where F is `$AGT`.

In [53]:
def first_column_starts(bwt):
    # F is just the BWT letters sorted
    first_column = sorted(bwt)

    # starts will map each letter to the first row where it appears in F
    starts = {}
    for position in range(len(first_column)):
        letter = first_column[position]
        if letter not in starts:
            starts[letter] = position
    return starts

print(first_column_starts("TG$A"))

{'$': 0, 'A': 1, 'G': 2, 'T': 3}


### Counting letters in L (rank)

After the search finds a range of rows, it has to move one letter to the left in the pattern. To do that I need to know how many times a letter appears in the last column L above a certain row. I call this the rank.

This works because the copies of a letter appear in the same order in L and in F. If a T has two T's above it in L, it is the third T in F as well. So the rank tells me which row in F that letter lands on.

The function below counts how many times a letter appears in the first `row` characters of L. I tested it on the BWT of GATTACA (`ACTGA$TA`), where I worked out the expected answers by hand.

In [58]:
def rank(bwt, letter, row):
    # count how many times `letter` appears in the first `row` characters of L
    count = 0
    for i in range(row):
        if bwt[i] == letter:
            count = count + 1
    return count

# tests on the GATTACA BWT
L = "ACTGA$TA"
print(rank(L, "A", 5))   # expect 2
print(rank(L, "T", 3))   # expect 1
print(rank(L, "A", 0))   # expect 0
print(rank(L, "A", 8))   # expect 3, the total number of A's

2
1
0
3


### Backward search

To count a pattern, I read it from its last letter to its first. I start with the range of all rows. For each letter, I use the starting row of that letter in F plus the rank in L to move the range to the rows where the pattern so far matches. If the range becomes empty, the pattern is not there. If I reach the start of the pattern, the number of rows left in the range is the number of occurrences.

In [49]:
def backward_search_count(bwt, pattern):
    starts = first_column_starts(bwt)

    # at the start, the range is every row of the table
    # top is the first row in the range, bottom is one past the last row
    top = 0
    bottom = len(bwt)

    # go through the pattern from its LAST letter to its FIRST
    for letter in reversed(pattern):

        # if the letter is not in the text at all, there are no matches
        if letter not in starts:
            return 0

        # narrow the range to rows where this letter comes before the current range
        top = starts[letter] + rank(bwt, letter, top)
        bottom = starts[letter] + rank(bwt, letter, bottom)

        # if the range is empty, the pattern is not in the text
        if top >= bottom:
            return 0

    # rows left in the range = number of occurrences
    return bottom - top


# tests on GATTACA, expected counts worked out by hand
test_bwt = "ACTGA$TA"
print(backward_search_count(test_bwt, "TA"))       
print(backward_search_count(test_bwt, "A"))         
print(backward_search_count(test_bwt, "GATTACA"))  
print(backward_search_count(test_bwt, "CC"))      

1
3
1
0


### Counting AGATA with the BWT and checking the result

Here I use the BWT of the reference to count AGATA and its reverse complement TATCT with backward search. I then compare these counts with the direct string search from earlier. If the two methods agree, that is evidence that the BWT search is working.

In [50]:
# count AGATA and its reverse complement using the BWT of the reference
bwt_count_pattern = backward_search_count(reference_bwt, pattern)
bwt_count_revcomp = backward_search_count(reference_bwt, reverse_complement)

# counts from the direct search done earlier
direct_count_pattern = len(pattern_positions)
direct_count_revcomp = len(reverse_complement_positions)

print("Pattern:", pattern)
print("BWT count:", bwt_count_pattern, "| direct count:", direct_count_pattern)
print("Reverse complement:", reverse_complement)
print("BWT count:", bwt_count_revcomp, "| direct count:", direct_count_revcomp)

# the two methods must agree
assert bwt_count_pattern == direct_count_pattern
assert bwt_count_revcomp == direct_count_revcomp
print("Both methods agree.")

Pattern: AGATA
BWT count: 5 | direct count: 5
Reverse complement: TATCT
BWT count: 3 | direct count: 3
Both methods agree.


## 11. Answer

AGATA occurs 5 times in the reference sequence (zamsref, 1989 bases). Its reverse complement, TATCT, occurs 3 times.

I counted both using backward search on the BWT of the reference, then checked them with a direct search along the sequence. Both methods gave the same counts. The direct search also counts overlapping matches: two AGATA matches start at positions 917 and 921 and share one A.

I searched for the reverse complement because the sequence of interest could be on either DNA strand. The 3 matches of TATCT mean AGATA appears 3 times on the opposite strand.

## 12. Limitations

I only tested this code on a short example and one small reference of 1,989 bases. I did not run it on large genomic data. It would not work well at that scale, for three reasons.

My BWT function builds and sorts every rotation of the sequence, which needs memory proportional to the length squared. Real tools build the BWT in a smarter way, from a suffix array. The inverse BWT is also slow for long sequences, because it re-sorts the whole table once for every letter.

My rank function counts from the start of L every time it is called, so each search step gets slower as the sequence grows. An FM-index stores counts at intervals to make this fast. The backward search also only gives a count, not the positions of the matches, so the direct search was used for positions. The code only accepts A, C, G and T, so a reference containing N would need extra handling.

This was my first time working with this algorithm. I understand the main idea, but I would need more time to understand the details fully, especially why rank and the first-column positions let the search move backwards correctly.

## 13. Use of Generative AI

I used Claude (Anthropic's AI assistant) while working on this task. It helped me understand how the BWT and backward search work, and it helped me write and debug parts of the code, especially the backward search functions in Section 10 and the worked example in Section 3. I ran and tested all the code myself, checked the BWT counts against a direct search, and I can explain each part.